In [1]:
# ==============================================================================
# SCRIPT 6 - FINAL LAYER ASSEMBLY FOR THE ARCGIS ONLINE DASHBOARD
# ENVIRONMENT SETUP
# ==============================================================================
%pip install geopandas pandas fiona pyarrow shapely -q
import os
import pandas as pd
import geopandas as gpd
import fiona

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 51.6 MB/s eta 0:00:00


In [ ]:
# ==============================================================================
# LOCATE THE PROJECT FOLDER
# ==============================================================================
# Finds the cloned repository (the folder that contains 03_scripts).
# Works locally, in Colab and elsewhere. Set the PROJECT_ROOT environment variable to override.
import os
from pathlib import Path

def find_project_root():
    if os.environ.get("PROJECT_ROOT"):
        return Path(os.environ["PROJECT_ROOT"])
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "03_scripts").is_dir():
            return folder
    colab_clone = Path("/content/rail-station-property-values")
    if colab_clone.is_dir():
        return colab_clone
    raise FileNotFoundError("Project folder not found. Run the notebook from inside the cloned repository, or set PROJECT_ROOT.")

PROJECT_ROOT = str(find_project_root())
print(f"Project folder: {PROJECT_ROOT}")

In [ ]:
# ==============================================================================
# STANDALONE LSOA IMD (IoD2025) HEATMAP LAYER
# ------------------------------------------------------------------------------
# Clips national LSOA boundaries to the 2km station buffers and attaches the
# IoD2025 deprivation decile already carried in the residential model-ready file.
# ==============================================================================
print("GENERATING LSOA IMD (IoD2025) HEATMAP LAYER")
BASE_DIR = PROJECT_ROOT

RAW_LSOA_PATH       = os.path.join(BASE_DIR, '01_raw_data/spatial/LSOA_2021/LSOA_2021.geojson')
STATION_BUFFER_PATH = os.path.join(BASE_DIR, '01_raw_data/Stations_and_2000m_Buffer/All_Stations_2000m_Buffer.geojson')
RES_PARQUET_PATH    = os.path.join(BASE_DIR, '02_processed_data/03_model_ready/regression_ready_residential.parquet')
LSOA_OUT_PATH       = os.path.join(BASE_DIR, '04_results/maps/01_lsoa_imd_2025.gpkg')
os.makedirs(os.path.dirname(LSOA_OUT_PATH), exist_ok=True)

# IMD lookup from the residential model-ready data (one row per LSOA in the study area)
df_res = pd.read_parquet(RES_PARQUET_PATH)
df_imd_lookup = df_res[['LSOA21CD', 'IMD_Decile']].drop_duplicates()
print(f"   -> {len(df_imd_lookup)} unique LSOA zones with IMD deciles in the study area.")

gdf_buffer = gpd.read_file(STATION_BUFFER_PATH).to_crs("EPSG:27700")
print("Loading national LSOA boundaries (this can take a minute)...")
gdf_lsoa = gpd.read_file(RAW_LSOA_PATH).to_crs("EPSG:27700")
if 'lsoa21cd' in gdf_lsoa.columns:
    gdf_lsoa.rename(columns={'lsoa21cd': 'LSOA21CD'}, inplace=True)

gdf_study_lsoa = gpd.clip(gdf_lsoa, gdf_buffer)
gdf_study_lsoa = gdf_study_lsoa.merge(df_imd_lookup, on='LSOA21CD', how='left')
gdf_study_lsoa = gdf_study_lsoa.dropna(subset=['IMD_Decile'])

gdf_web = gdf_study_lsoa.to_crs("EPSG:4326")
gdf_web[['LSOA21CD', 'IMD_Decile', 'geometry']].to_file(
    LSOA_OUT_PATH, layer='01_lsoa_imd_2025', driver='GPKG')
print(f"\nLSOA IMD layer written to {LSOA_OUT_PATH}")

GENERATING LSOA IMD (IoD2025) HEATMAP LAYER
   -> 227 unique LSOA zones with IMD deciles in the study area.
Loading national LSOA boundaries (this can take a minute)...

LSOA IMD layer written to /content/drive/MyDrive/MSc_Dissertation_Project/04_results/maps/01_lsoa_imd_2025.gpkg


In [ ]:
# ==============================================================================
# ASSEMBLE THE ARCGIS DASHBOARD GEOPACKAGE
# ------------------------------------------------------------------------------
# Reads the Script 5 results (with premium fields) and packs every layer
# the dashboard needs into one WGS84 GeoPackage.
# The per-property "premium" shown to users is
# Zone_Premium_GBP - the DiD-estimated premium applied to treated near-zone,
# post-opening properties.
# ==============================================================================
print("ASSEMBLING ARCGIS MASTER DASHBOARD DATABASE")
BASE_DIR = PROJECT_ROOT

RES_RESULTS_PATH = os.path.join(BASE_DIR, '04_results/residential_with_premiums.parquet')
COMM_RESULTS_PATH= os.path.join(BASE_DIR, '04_results/commercial_with_premiums.parquet')
LSOA_IMD_PATH    = os.path.join(BASE_DIR, '04_results/maps/01_lsoa_imd_2025.gpkg')
WALK_NET_PATH    = os.path.join(BASE_DIR, '02_processed_data/04_network/devon_walk_network.gpkg')
RAIL_NET_PATH    = os.path.join(BASE_DIR, '02_processed_data/04_network/rail_network_lines.gpkg')
STATIONS_PATH    = os.path.join(BASE_DIR, '02_processed_data/04_network/station_nodes.gpkg')
ISOCHRONES_PATH  = os.path.join(BASE_DIR, '02_processed_data/04_network/isochrone_bands.gpkg')
STATIC_DB_PATH   = os.path.join(BASE_DIR, '02_processed_data/static_master_layer.gpkg')
DASHBOARD_DB     = os.path.join(BASE_DIR, '04_results/maps/ArcGIS_Dashboard_Results.gpkg')

if os.path.exists(DASHBOARD_DB):
    os.remove(DASHBOARD_DB)
    print("Cleared old dashboard database. Starting fresh.")

is_first_layer = True
def write_to_dashboard(gdf, layer_name):
    global is_first_layer
    # cast bools to int (GPKG-friendly)
    for col in gdf.columns:
        if gdf[col].dtype == bool:
            gdf[col] = gdf[col].astype(int)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    gdf.to_file(DASHBOARD_DB, layer=layer_name, driver='GPKG',
                mode=('w' if is_first_layer else 'a'))
    print(f"   packed: {layer_name}")
    is_first_layer = False

# Human-readable field names. Geometry source columns (east1m/north1m) are kept.
master_rename_dict = {
    'Postcode': 'Postcode',
    'Total_Floor_Area': 'Floor Area (sqm)',
    'CommonName': 'Nearest Station',
    'walk_dist_m': 'Walking Distance (m)',
    'Distance_Band': 'Distance Band',
    'IMD_Decile': 'Deprivation Decile (1-10)',
    'Epoch': 'Market Epoch (Phase)',
    'Treatment_Zone': 'Treatment Zone',
    # Premium / model fields
    'Zone_Premium_GBP': 'Estimated Transit Premium (£)',
    'Predicted_Price_GBP': 'Model Predicted Price (£)',
    'Predicted_RV_GBP': 'Predicted Rateable Value (£)',
    'Model_Residual_GBP': 'Model Residual (£, unexplained)',
    'Service_Intensity_SEII': 'Station Service Intensity',
    'Service_Intensity_Epoch': 'Service Intensity (epoch-aware)',
    # Residential specifics
    'UPRN': 'Unique Property Reference Number (UPRN)',
    'Street': 'Street Address',
    'Property_Type': 'Property Type',
    'EPC_Rating': 'EPC Rating',
    'Is_New_Build': 'New Build (1=Yes, 0=No)',
    'Date_of_Transfer': 'Transaction Date',
    'Price': 'Actual Price (£)',
    'Price_Normalised': 'HPI-Adjusted Price (£)',
    # Commercial specifics
    'Assessment_UARN': 'Unique Assessment Reference Number (UARN)',
    'Property_Sector': 'Commercial Sector',
    'Year': 'Valuation Year',
    'Rateable_Value': 'Actual Rateable Value (£)',
    'Sector_Group': 'Sector Category',
}

def points_layer(path, layer_name, label):
    if not os.path.exists(path):
        print(f"   MISSING: {path}"); return
    print(f"Converting {label} results to web points...")
    df = pd.read_parquet(path)
    g = gpd.GeoDataFrame(df.rename(columns=master_rename_dict),
                         geometry=gpd.points_from_xy(df['east1m'], df['north1m']),
                         crs="EPSG:27700")
    write_to_dashboard(g, layer_name)

points_layer(RES_RESULTS_PATH, '05_residential_results', 'residential')
points_layer(COMM_RESULTS_PATH, '05_commercial_results', 'commercial')

context_layers = [
    ('01_lsoa_imd_2025', LSOA_IMD_PATH, None),
    ('01_rail_network', RAIL_NET_PATH, None),
    ('02_treatment_and_control_stations', STATIONS_PATH, None),
    ('02_pedestrian_routing_network', WALK_NET_PATH, None),
    ('02_isochrone_bands', ISOCHRONES_PATH, None),
    ('01_devon_boundary', STATIC_DB_PATH, '01_devon_boundary'),
    ('04_market_liquidity_hexgrid', STATIC_DB_PATH, '04_market_liquidity_hexgrid'),
]
print("\nProcessing context layers...")
for target_name, path, internal_layer in context_layers:
    if not os.path.exists(path):
        print(f"   skipping {target_name} (not found)"); continue
    try:
        g = gpd.read_file(path, layer=internal_layer) if internal_layer else gpd.read_file(path)
        # pedestrian network gpkg stores edges in an 'edges' layer
        if target_name == '02_pedestrian_routing_network' and 'geometry' not in g.columns:
            g = gpd.read_file(path, layer='edges')
        write_to_dashboard(g, target_name)
    except Exception as e:
        print(f"   error packing {target_name}: {e}")

print(f"\nDASHBOARD COMPLETE -> {DASHBOARD_DB}")

ASSEMBLING ARCGIS MASTER DASHBOARD DATABASE
Cleared old dashboard database. Starting fresh.
Converting residential results to web points...
   packed: 05_residential_results
Converting commercial results to web points...
   packed: 05_commercial_results

Processing context layers...
   packed: 01_lsoa_imd_2025
   packed: 01_rail_network
   packed: 02_treatment_and_control_stations


/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'devon_walk_network.gpkg': 'edges' (default), 'nodes'. Specify layer parameter to avoid this warning.
  result = read_func(


   packed: 02_pedestrian_routing_network
   packed: 02_isochrone_bands
   packed: 01_devon_boundary
   packed: 04_market_liquidity_hexgrid

DASHBOARD COMPLETE -> /content/drive/MyDrive/MSc_Dissertation_Project/04_results/maps/ArcGIS_Dashboard_Results.gpkg


In [ ]:
# ==============================================================================
# EXPORT GEOJSON FILES FROM DASHBOARD GEOPACKAGE FOR ARCGIS ONLINE
# ------------------------------------------------------------------------------
# Reads from the master GeoPackage.
# Adds Treated field to stations layer for treated vs control symbology.
# Remaps residential Property Type from single letters to full names.
# ==============================================================================

GEOJSON_DIR = os.path.join(BASE_DIR, '04_results/maps/individual_layers/geojson')
os.makedirs(GEOJSON_DIR, exist_ok=True)

LAYERS = [
    '05_residential_results',
    '05_commercial_results',
    '01_lsoa_imd_2025',
    '01_rail_network',
    '02_treatment_and_control_stations',
    '02_isochrone_bands',
    '01_devon_boundary',
]

print("Exporting GeoJSON files from dashboard GeoPackage...\n")

for layer in LAYERS:
    try:
        gdf = gpd.read_file(DASHBOARD_DB, layer=layer)

        # Add Treated field to stations for treated vs control symbology
        if layer == '02_treatment_and_control_stations':
            gdf['Treated'] = gdf['CommonName'].str.contains(
                'Okehampton|Marsh Barton', na=False
            ).astype(int)
            print(f"   {layer}: done")

        elif layer == '05_residential_results':
            gdf = gdf.drop_duplicates()
            prop_type_map = {
                'D': 'Detached',
                'S': 'Semi-detached',
                'T': 'Terraced',
                'F': 'Flat',
            }
            if 'Property Type' in gdf.columns:
                gdf['Property Type'] = gdf['Property Type'].map(prop_type_map).fillna(gdf['Property Type'])
            print(f"   {layer}: done")

        elif layer == '05_commercial_results':
            gdf = gdf.drop_duplicates()
            print(f"   {layer}: done")

        else:
            print(f"   {layer}: done")

        out_path = os.path.join(GEOJSON_DIR, f"{layer}.geojson")
        if os.path.exists(out_path):
            os.remove(out_path)
        gdf.to_file(out_path, driver='GeoJSON')

    except Exception as e:
        print(f"   ERROR on {layer}: {e}")

print(f"\nDone. Files at: {GEOJSON_DIR}")

Exporting GeoJSON files from dashboard GeoPackage...

   05_residential_results: done
   05_commercial_results: done
   01_lsoa_imd_2025: done
   01_rail_network: done
   02_treatment_and_control_stations: done
   02_isochrone_bands: done
   01_devon_boundary: done

Done. Files at: /content/drive/MyDrive/MSc_Dissertation_Project/04_results/maps/individual_layers/geojson


In [ ]:
# ==============================================================================
# EXPORT 05_commercial_results.geojson as CSV FOR ARCGIS ONLINE
# ------------------------------------------------------------------------------
# Other formats, including shapefile and Geopackage refuse to load
# ==============================================================================

import geopandas as gpd

GEOJSON_DIR = os.path.join(BASE_DIR, '04_results/maps/individual_layers/geojson')
src_path    = os.path.join(GEOJSON_DIR, '05_commercial_results.geojson')
out_path    = os.path.join(GEOJSON_DIR, '05_commercial_results.csv')

gdf = gpd.read_file(src_path)
gdf['latitude']  = gdf.geometry.y
gdf['longitude'] = gdf.geometry.x
df = gdf.drop(columns='geometry')

df.to_csv(out_path, index=False)
print(f"Commercial CSV: Done -> {out_path}")

Commercial CSV: Done -> /content/drive/MyDrive/MSc_Dissertation_Project/04_results/maps/individual_layers/geojson/05_commercial_results.csv


In [ ]:
# ==============================================================================
# REPROJECT CARTOGRAPHIC LAYERS TO BRITISH NATIONAL GRID (EPSG:27700)
# ==============================================================================

BNG    = "EPSG:27700"
BNG_DB = os.path.join(BASE_DIR, '04_results/maps/Cartographic_Layers_BNG.gpkg')

CARTO_LAYERS = [
    '05_residential_results',
    '05_commercial_results',
    '02_isochrone_bands',
    '02_treatment_and_control_stations',
    '01_rail_network',
]

if os.path.exists(BNG_DB):
    os.remove(BNG_DB)
    print("Cleared existing BNG file.")

first = True
for layer in CARTO_LAYERS:
    print(f"Reprojecting  {layer}...", end="  ")
    gdf     = gpd.read_file(DASHBOARD_DB, layer=layer)
    gdf_bng = gdf.to_crs(BNG)
    gdf_bng.to_file(BNG_DB, layer=layer, driver='GPKG', mode=('w' if first else 'a'))
    print("done")
    first = False

print(f"\nBNG GeoPackage written to:\n{BNG_DB}")

Cleared existing BNG file.
Reprojecting  05_residential_results...  done
Reprojecting  05_commercial_results...  done
Reprojecting  02_isochrone_bands...  done
Reprojecting  02_treatment_and_control_stations...  done
Reprojecting  01_rail_network...  done

BNG GeoPackage written to:
/content/drive/MyDrive/MSc_Dissertation_Project/04_results/maps/Cartographic_Layers_BNG.gpkg


In [ ]:
# ==============================================================================
# DEVON COOKIE CUTTER MASK FOR DASHBOARD BASEMAP
# ==============================================================================
from shapely.geometry import box

# Load Devon boundary
devon = gpd.read_file(STATIC_DB_PATH, layer='01_devon_boundary').to_crs("EPSG:4326")
devon_union = devon.geometry.union_all()

# Create a large bounding box covering the whole UK
world_box = box(-12, 48, 4, 62)

# Subtract Devon from the bounding box — the result covers everything except Devon
mask = world_box.difference(devon_union)

gdf_mask = gpd.GeoDataFrame({'geometry': [mask]}, crs="EPSG:4326")

out_path = os.path.join(GEOJSON_DIR, 'devon_cookie_cutter.geojson')
if os.path.exists(out_path):
    os.remove(out_path)
gdf_mask.to_file(out_path, driver='GeoJSON')
print(f"Cookie cutter mask exported: {out_path}")

Cookie cutter mask exported: /content/drive/MyDrive/MSc_Dissertation_Project/04_results/maps/individual_layers/geojson/devon_cookie_cutter.geojson
